# Nested Multi-Agent Pattern

The **nested multi-agent pattern** treats an entire multi-agent workflow as **one step** inside a larger workflow. The outer orchestrator sees a **black-box sub-workflow**; the inner orchestrator runs its own specialists in sequence (or with loops) and returns a **bounded result** to the parent.

```
OUTER:  Triage ──► [ Research Sub-flow ] ──► [ Content Sub-flow ] ──► Publish
                         │                           │
INNER:            Changelog ──► Runbook ──► Merge    Writer ──► Critic ↺
```

Nesting is **composition**, not the same as a **hierarchical supervisor** (which routes by org level). Here you package a pipeline fragment so it can be reused, tested, and reasoned about independently.

This notebook implements **ReleaseFlow** with two nested sub-workflows in plain Python: state boundaries, hand-off contracts, depth-aware tracing, error propagation, and isolation tests. No frameworks or prior notebooks are required.

In [ ]:
"""
ReleaseFlow — nested multi-agent workflow for v2.4.0 announcements.
"""

import json
import os
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Protocol

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


CHANGELOG: list[dict[str, str]] = [
    {"version": "2.4.0", "item": "Added bulk export endpoint for reports"},
    {"version": "2.4.0", "item": "Fixed session timeout on mobile browsers"},
    {"version": "2.4.0", "item": "Deprecated legacy v1 webhook format"},
]

RUNBOOKS: list[dict[str, str]] = [
    {"id": "deploy-200", "title": "Deploy checklist",
     "text": "Canary 10%, monitor 15 min, then full rollout."},
    {"id": "comm-201", "title": "Customer communication standards",
     "text": "Lead with user benefit, list breaking changes separately, include #releases."},
]

STYLE_GUIDE = {"max_words": 200}
PUBLISHED: list[dict[str, Any]] = []

print(f"Corpus: {len(CHANGELOG)} changelog | {len(RUNBOOKS)} runbooks")

---

## 1. Pattern Definition

| Property | Nested pattern |
|----------|----------------|
| **Structure** | Parent workflow contains child workflows as steps |
| **Encapsulation** | Inner agents hidden from outer orchestrator |
| **Interface** | Sub-workflow exposes `input → output` contract to parent |
| **State** | Often scoped: inner scratch state + exported artifacts |
| **Reuse** | Same sub-flow in multiple parent pipelines |

**Analogy:** A function that calls other functions. The outer pipeline calls `research_subflow(goal)` without knowing internal steps.

---

## 2. Nested vs Flat vs Hierarchical

| Pattern | Shape | When |
|---------|-------|------|
| **Flat sequential** | A → B → C → D (one list) | Small pipeline, ≤5 stages |
| **Nested** | Outer(A → [inner x→y→z] → B) | Reusable phases, team ownership per sub-flow |
| **Hierarchical supervisor** | Tree of routers by domain | Many workers, org-aligned delegation |

Nest when a **phase** is complex enough to deserve its own orchestrator and tests — not when you simply have many linear steps (flatten those first).

---

## 3. ReleaseFlow Topology

```
┌─────────────────────────────────────────────────────────────────┐
│ OUTER: NestedReleasePipeline                                     │
│                                                                  │
│  triage ──► ┌──────────────────┐ ──► ┌──────────────────┐ ──► publish
│             │ ResearchSubflow  │     │ ContentSubflow   │
│             │  changelog_agent │     │  writer_agent    │
│             │  runbook_agent   │     │  critic_agent ↺  │
│             │  merge_agent     │     │  formatter_agent │
│             └──────────────────┘     └──────────────────┘
└─────────────────────────────────────────────────────────────────┘
```

| Boundary | Parent reads | Sub-flow exports |
|----------|--------------|------------------|
| Research → Content | `research_bundle` | version, changes, comm_rules, deploy_notes |
| Content → Publish | `formatted_draft`, `approved` | customer-ready text + approval flag |

In [ ]:
@dataclass
class TraceEvent:
    depth: int
    workflow: str
    step: str
    agent: str
    status: str
    detail: str = ""


@dataclass
class WorkflowContext:
    """Shared outer state; sub-flows write only to declared export keys."""
    run_id: str
    goal: str
    artifacts: dict[str, Any] = field(default_factory=dict)
    trace: list[TraceEvent] = field(default_factory=list)
    status: str = "running"

    def log_trace(self, depth: int, workflow: str, step: str, agent: str, status: str = "ok", detail: str = "") -> None:
        self.trace.append(TraceEvent(depth, workflow, step, agent, status, detail))


@dataclass
class SubflowResult:
    success: bool
    exports: dict[str, Any]
    inner_trace: list[TraceEvent] = field(default_factory=list)
    error: str = ""

---

## 4. SubWorkflow Protocol

Every nested workflow implements the same interface so the parent can treat them uniformly.

In [ ]:
class SubWorkflow(Protocol):
    name: str

    def run(self, ctx: WorkflowContext, *, depth: int = 1) -> SubflowResult:
        ...


SUBFLOW_EXPORT_KEYS = {
    "research": ["research_bundle"],
    "content": ["formatted_draft", "approved"],
}

print("SubWorkflow protocol defined")

---

## 5. Inner Agents — Research Phase

Three specialists only visible inside `ResearchSubflow`.

In [ ]:
class ChangelogAgent:
    name = "changelog_scout"

    def run(self, goal: str) -> dict[str, Any]:
        return {"version": "2.4.0", "changes": [c["item"] for c in CHANGELOG]}


class RunbookAgent:
    name = "runbook_scout"

    def run(self, goal: str) -> dict[str, Any]:
        return {
            "deploy_notes": RUNBOOKS[0]["text"],
            "comm_rules": RUNBOOKS[1]["text"],
        }


class ResearchMergerAgent:
    name = "research_merger"

    def run(self, changelog: dict[str, Any], runbook: dict[str, Any]) -> dict[str, Any]:
        return {
            "version": changelog["version"],
            "changes": changelog["changes"],
            "deploy_notes": runbook["deploy_notes"],
            "comm_rules": runbook["comm_rules"],
            "source_count": len(changelog["changes"]) + 2,
        }


print("Research inner agents: changelog_scout, runbook_scout, research_merger")

---

## 6. ResearchSubflow — Nested Workflow Implementation

In [ ]:
class ResearchSubflow:
    name = "research"

    def __init__(self) -> None:
        self.changelog = ChangelogAgent()
        self.runbook = RunbookAgent()
        self.merger = ResearchMergerAgent()

    def run(self, ctx: WorkflowContext, *, depth: int = 1) -> SubflowResult:
        inner_trace: list[TraceEvent] = []
        wf = self.name

        try:
            cl = self.changelog.run(ctx.goal)
            inner_trace.append(TraceEvent(depth, wf, "fetch_changelog", self.changelog.name, "ok"))
            ctx.log_trace(depth, wf, "fetch_changelog", self.changelog.name)

            rb = self.runbook.run(ctx.goal)
            inner_trace.append(TraceEvent(depth, wf, "fetch_runbooks", self.runbook.name, "ok"))
            ctx.log_trace(depth, wf, "fetch_runbooks", self.runbook.name)

            bundle = self.merger.run(cl, rb)
            inner_trace.append(TraceEvent(depth, wf, "merge", self.merger.name, "ok", f"sources={bundle['source_count']}"))
            ctx.log_trace(depth, wf, "merge", self.merger.name, detail=f"sources={bundle['source_count']}")

            return SubflowResult(True, {"research_bundle": bundle}, inner_trace)
        except Exception as exc:
            inner_trace.append(TraceEvent(depth, wf, "error", "subflow", "error", str(exc)))
            return SubflowResult(False, {}, inner_trace, error=str(exc))


research_demo = ResearchSubflow().run(WorkflowContext(run_id="demo", goal="test"), depth=1)
print("Research export keys:", list(research_demo.exports.keys()))
print("Bundle version:", research_demo.exports["research_bundle"]["version"])

---

## 7. Inner Agents — Content Phase

Writer, critic (with loop), and formatter — encapsulated in `ContentSubflow`.

In [ ]:
class ContentWriterAgent:
    name = "writer"

    def run(self, bundle: dict[str, Any], feedback: str = "") -> str:
        draft = (
            f"Release v{bundle['version']} is now available.\n\n"
            f"What's new: " + "; ".join(bundle["changes"]) + ".\n\n"
            f"Breaking changes: Legacy v1 webhook format is deprecated.\n\n"
            f"Deploy note: {bundle['deploy_notes'][:50]}...\n\n"
            f"Questions? Contact #releases."
        )
        if feedback and "too long" in feedback:
            draft = "\n".join(draft.split("\n")[:4]) + "\n\nQuestions? Contact #releases."
        return draft


class ContentCriticAgent:
    name = "critic"

    def run(self, draft: str) -> tuple[bool, str]:
        issues = []
        if "v2.4" not in draft:
            issues.append("missing version")
        if "Breaking changes" not in draft:
            issues.append("missing breaking section")
        if "#releases" not in draft:
            issues.append("missing support channel")
        if len(draft.split()) > STYLE_GUIDE["max_words"]:
            issues.append(f"too long ({len(draft.split())} words)")
        return (not issues, ", ".join(issues))


class FormatterAgent:
    name = "formatter"

    def run(self, draft: str) -> str:
        return draft.strip() + "\n\n---\nShopCo Release Team"


print("Content inner agents: writer, critic, formatter")

In [ ]:
class ContentSubflow:
    name = "content"

    def __init__(self, max_revisions: int = 2) -> None:
        self.writer = ContentWriterAgent()
        self.critic = ContentCriticAgent()
        self.formatter = FormatterAgent()
        self.max_revisions = max_revisions

    def run(self, ctx: WorkflowContext, *, depth: int = 1) -> SubflowResult:
        bundle = ctx.artifacts.get("research_bundle")
        if not bundle:
            return SubflowResult(False, {}, [], error="missing research_bundle from parent")

        inner_trace: list[TraceEvent] = []
        wf = self.name
        feedback = ""
        approved = False
        draft = ""

        for revision in range(self.max_revisions + 1):
            draft = self.writer.run(bundle, feedback)
            inner_trace.append(TraceEvent(depth, wf, "write", self.writer.name, "ok", f"rev={revision}"))
            ctx.log_trace(depth, wf, "write", self.writer.name, detail=f"rev={revision}")

            approved, feedback = self.critic.run(draft)
            status = "ok" if approved else "rejected"
            inner_trace.append(TraceEvent(depth, wf, "review", self.critic.name, status, feedback))
            ctx.log_trace(depth, wf, "review", self.critic.name, status=status, detail=feedback)

            if approved:
                break

        if not approved:
            return SubflowResult(False, {"approved": False, "review_feedback": feedback}, inner_trace,
                                 error="max revisions exceeded")

        formatted = self.formatter.run(draft)
        inner_trace.append(TraceEvent(depth, wf, "format", self.formatter.name, "ok"))
        ctx.log_trace(depth, wf, "format", self.formatter.name)

        return SubflowResult(True, {"formatted_draft": formatted, "approved": True}, inner_trace)


ctx_test = WorkflowContext(run_id="t", goal="test")
ctx_test.artifacts["research_bundle"] = research_demo.exports["research_bundle"]
content_demo = ContentSubflow().run(ctx_test)
print("Content approved:", content_demo.exports.get("approved"))
print("Formatted preview:", content_demo.exports.get("formatted_draft", "")[:100], "...")

---

## 8. Outer NestedReleasePipeline

The parent orchestrator: triage → sub-flows → publish. It never calls inner agents directly.

In [ ]:
class TriageAgent:
    name = "triage"

    def run(self, ctx: WorkflowContext) -> None:
        ctx.artifacts["release_type"] = "minor" if "2.4" in ctx.goal else "unknown"
        ctx.artifacts["priority"] = "normal"
        ctx.log_trace(0, "outer", "triage", self.name, detail=f"type={ctx.artifacts['release_type']}")


class PublishAgent:
    name = "publisher"

    def run(self, ctx: WorkflowContext) -> bool:
        if not ctx.artifacts.get("approved"):
            ctx.log_trace(0, "outer", "publish", self.name, status="blocked")
            return False
        ann = {
            "id": f"ANN-{uuid.uuid4().hex[:6].upper()}",
            "body": ctx.artifacts.get("formatted_draft", ""),
            "published_at": datetime.now(timezone.utc).isoformat(),
        }
        PUBLISHED.append(ann)
        ctx.artifacts["announcement_id"] = ann["id"]
        ctx.log_trace(0, "outer", "publish", self.name, detail=ann["id"])
        return True


@dataclass
class NestedPipelineResult:
    run_id: str
    status: str
    published: bool
    announcement_id: str
    subflows_run: list[str]
    trace_depth_max: int
    trace: list[TraceEvent]


class NestedReleasePipeline:
    """Outer orchestrator composing nested sub-workflows."""

    def __init__(self) -> None:
        self.triage = TriageAgent()
        self.research_subflow = ResearchSubflow()
        self.content_subflow = ContentSubflow(max_revisions=2)
        self.publisher = PublishAgent()

    def _apply_exports(self, ctx: WorkflowContext, result: SubflowResult, subflow_name: str) -> bool:
        if not result.success:
            ctx.status = "failed"
            ctx.log_trace(0, "outer", subflow_name, "subflow", status="error", detail=result.error)
            return False
        allowed = SUBFLOW_EXPORT_KEYS.get(subflow_name, [])
        for key in allowed:
            if key in result.exports:
                ctx.artifacts[key] = result.exports[key]
        ctx.log_trace(0, "outer", subflow_name, "subflow", detail=f"exported {allowed}")
        return True

    def run(self, goal: str, *, force_long_draft: bool = False) -> NestedPipelineResult:
        ctx = WorkflowContext(run_id=f"nested_{uuid.uuid4().hex[:8]}", goal=goal)
        subflows_run: list[str] = []

        self.triage.run(ctx)

        r = self.research_subflow.run(ctx, depth=1)
        subflows_run.append(self.research_subflow.name)
        if not self._apply_exports(ctx, r, "research"):
            return self._finalize(ctx, subflows_run, published=False)

        if force_long_draft:
            c = self._run_forced_revision(ctx)
        else:
            c = self.content_subflow.run(ctx, depth=1)
        subflows_run.append(self.content_subflow.name)
        if not self._apply_exports(ctx, c, "content"):
            return self._finalize(ctx, subflows_run, published=False)

        published = self.publisher.run(ctx)
        ctx.status = "published" if published else "blocked"
        return self._finalize(ctx, subflows_run, published=published)

    def _run_forced_revision(self, ctx: WorkflowContext) -> SubflowResult:
        """Demo path: inject overlong draft to exercise critic loop inside content subflow."""
        sub = ContentSubflow(max_revisions=2)
        bundle = ctx.artifacts["research_bundle"]
        inner: list[TraceEvent] = []
        draft = ContentWriterAgent().run(bundle) + " word" * 120
        inner.append(TraceEvent(1, "content", "write", "writer", "ok", "forced long"))
        approved, feedback = ContentCriticAgent().run(draft)
        inner.append(TraceEvent(1, "content", "review", "critic", "rejected", feedback))
        if not approved:
            draft = ContentWriterAgent().run(bundle, feedback)
            inner.append(TraceEvent(1, "content", "write", "writer", "ok", "rev=1"))
            approved, _ = ContentCriticAgent().run(draft)
            inner.append(TraceEvent(1, "content", "review", "critic", "ok" if approved else "rejected"))
        formatted = FormatterAgent().run(draft)
        inner.append(TraceEvent(1, "content", "format", "formatter", "ok"))
        return SubflowResult(approved, {"formatted_draft": formatted, "approved": approved}, inner)

    def _finalize(self, ctx: WorkflowContext, subflows: list[str], published: bool) -> NestedPipelineResult:
        depths = [e.depth for e in ctx.trace] or [0]
        return NestedPipelineResult(
            run_id=ctx.run_id,
            status=ctx.status,
            published=published,
            announcement_id=ctx.artifacts.get("announcement_id", ""),
            subflows_run=subflows,
            trace_depth_max=max(depths),
            trace=ctx.trace,
        )


nested = NestedReleasePipeline()
print("NestedReleasePipeline ready")

---

## 9. Full Nested Run — Happy Path

In [ ]:
result = nested.run("Produce v2.4.0 customer release announcement")

print(f"Run: {result.run_id}")
print(f"Status: {result.status} | Published: {result.published}")
print(f"Sub-flows: {result.subflows_run}")
print(f"Announcement: {result.announcement_id}")
print(f"Max trace depth: {result.trace_depth_max}")
print("\nTrace (depth-aware):")
for e in result.trace:
    indent = "  " * e.depth
    print(f"{indent}[d={e.depth}] {e.workflow}/{e.step} ({e.agent}) — {e.status} {e.detail}")

---

## 10. Revision Inside Nested Content Sub-flow

In [ ]:
revision_result = nested.run("v2.4.0 with nested revision", force_long_draft=True)
content_steps = [e for e in revision_result.trace if e.workflow == "content"]
print(f"Published: {revision_result.published}")
print(f"Content sub-flow steps: {len(content_steps)}")
for e in content_steps:
    print(f"  {e.step} ({e.agent}) — {e.status} {e.detail}")

---

## 11. Testing Sub-flows in Isolation

A major benefit of nesting: **unit test inner workflows** without running the outer pipeline.

In [ ]:
def test_research_subflow_isolated() -> dict[str, Any]:
    ctx = WorkflowContext(run_id="iso_r", goal="isolated research test")
    result = ResearchSubflow().run(ctx, depth=1)
    bundle = result.exports.get("research_bundle", {})
    return {
        "success": result.success,
        "version": bundle.get("version"),
        "change_count": len(bundle.get("changes", [])),
        "inner_steps": len(result.inner_trace),
    }


def test_content_subflow_requires_bundle() -> dict[str, Any]:
    ctx = WorkflowContext(run_id="iso_c", goal="missing bundle")
    result = ContentSubflow().run(ctx)
    return {"success": result.success, "error": result.error}


print("Research isolated:", test_research_subflow_isolated())
print("Content without bundle:", test_content_subflow_requires_bundle())

---

## 12. Error Propagation — Parent Handles Child Failure

If a sub-flow fails, the parent must **not** continue to downstream steps.

In [ ]:
class FailingResearchSubflow(ResearchSubflow):
    def run(self, ctx: WorkflowContext, *, depth: int = 1) -> SubflowResult:
        return SubflowResult(False, {}, [], error="simulated changelog API timeout")


class BrokenNestedPipeline(NestedReleasePipeline):
    def __init__(self) -> None:
        super().__init__()
        self.research_subflow = FailingResearchSubflow()


fail = BrokenNestedPipeline().run("failure demo")
print(f"Status: {fail.status} | Published: {fail.published}")
print("Sub-flows attempted:", fail.subflows_run)
error_events = [e for e in fail.trace if e.status == "error"]
print("Error events:", [(e.workflow, e.detail) for e in error_events])

---

## 13. Nesting Depth and Limits

| Depth | Example | Risk |
|-------|---------|------|
| 0 | Outer triage, publish | — |
| 1 | ResearchSubflow, ContentSubflow | Manageable |
| 2 | Sub-flow inside sub-flow | Harder traces, brittle contracts |
| 3+ | Rarely justified | Debugging nightmare |

**Rule of thumb:** Prefer **depth ≤ 2**. If you need more, extract a service or flatten.

In [ ]:
MAX_RECOMMENDED_DEPTH = 2

def check_nesting_depth(trace: list[TraceEvent]) -> dict[str, Any]:
    max_d = max((e.depth for e in trace), default=0)
    return {
        "max_depth": max_d,
        "within_limit": max_d <= MAX_RECOMMENDED_DEPTH,
        "events_by_depth": {d: sum(1 for e in trace if e.depth == d) for d in range(max_d + 1)},
    }


print("Happy path depth check:", check_nesting_depth(result.trace))

---

## 14. Flat vs Nested — Same Work, Different Structure

| | Flat sequential (7 agents) | Nested (2 sub-flows + triage + publish) |
|--|---------------------------|----------------------------------------|
| **Orchestrator complexity** | One long list | Parent + 2 sub-orchestrators |
| **Team ownership** | Shared file | Research team owns `ResearchSubflow` |
| **Testing** | Stage fixtures | Sub-flow isolation tests |
| **Reuse** | Copy-paste stages | Import sub-flow elsewhere |
| **Trace** | Flat list | Depth-indented tree |

In [ ]:
FLAT_AGENT_ORDER = [
    "triage", "changelog_scout", "runbook_scout", "research_merger",
    "writer", "critic", "formatter", "publisher",
]
NESTED_STRUCTURE = {
    "outer": ["triage", "research_subflow", "content_subflow", "publisher"],
    "research_subflow": ["changelog_scout", "runbook_scout", "research_merger"],
    "content_subflow": ["writer", "critic", "formatter"],
}

flat_count = len(FLAT_AGENT_ORDER)
nested_leaf_count = sum(len(v) for k, v in NESTED_STRUCTURE.items() if k != "outer") + 2
print(f"Flat stages: {flat_count} | Nested leaf agents (approx): {nested_leaf_count}")
print("Nested outer steps:", NESTED_STRUCTURE["outer"])

---

## 15. Export Boundary Enforcement

Sub-flows should only expose **allow-listed keys** — inner scratch data must not leak.

In [ ]:
def apply_exports_strict(ctx: WorkflowContext, result: SubflowResult, subflow_name: str) -> list[str]:
    allowed = set(SUBFLOW_EXPORT_KEYS.get(subflow_name, []))
    applied = []
    for key, value in result.exports.items():
        if key in allowed:
            ctx.artifacts[key] = value
            applied.append(key)
    return applied


mock_result = SubflowResult(True, {
    "research_bundle": {"version": "2.4.0"},
    "internal_scratch": "should not leak",
})
ctx = WorkflowContext(run_id="x", goal="x")
applied = apply_exports_strict(ctx, mock_result, "research")
print("Applied keys:", applied)
print("Leaked internal_scratch:", "internal_scratch" in ctx.artifacts)

---

## 16. Anti-Patterns

| Anti-pattern | Problem | Fix |
|--------------|---------|-----|
| **Deep nesting (4+)** | Unreadable traces | Flatten or extract microservice |
| **Leaky inner state** | Parent depends on scratch keys | Allow-list exports |
| **No sub-flow interface** | Cannot test or reuse | `SubWorkflow` protocol |
| **Parent calls inner agents** | Breaks encapsulation | Only call `subflow.run()` |
| **Ignoring child errors** | Publish after failed research | Fail-fast in `_apply_exports` |

---

## 17. Optional Live LLM — Content Sub-flow Writer Only

In [ ]:
USE_LIVE_LLM = False


class LiveContentWriter(ContentWriterAgent):
    def run(self, bundle: dict[str, Any], feedback: str = "") -> str:
        if not USE_LIVE_LLM:
            return super().run(bundle, feedback)
        try:
            from openai import OpenAI
            client = OpenAI(api_key=OPENAI_API_KEY)
            prompt = f"Write release announcement for v{bundle['version']}. Changes: {bundle['changes']}. Feedback: {feedback or 'none'}"
            resp = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": prompt}], temperature=0.3)
            return resp.choices[0].message.content or ""
        except Exception as exc:
            return f"[LLM error: {exc}]"


if USE_LIVE_LLM:
    pipeline = NestedReleasePipeline()
    pipeline.content_subflow.writer = LiveContentWriter()
    print(pipeline.run("Live nested demo").status)
else:
    print("Offline mode — nesting works without any LLM SDK.")
    print(f"Published count: {len(PUBLISHED)}")

---

## 18. Quiz

1. What does it mean for a sub-workflow to be a "black box" to the parent?
2. How is nesting different from a hierarchical supervisor?
3. Why allow-list export keys from sub-flows?
4. What should the parent do when a child sub-flow returns `success=False`?
5. Why test sub-flows in isolation?

---

## 19. Summary

| Concept | Takeaway |
|---------|----------|
| Nested pattern | Multi-agent workflow as one parent step |
| Encapsulation | Parent calls `subflow.run()`, not inner agents |
| Export contract | Allow-listed keys cross the boundary |
| ReleaseFlow | `ResearchSubflow` + `ContentSubflow` inside `NestedReleasePipeline` |
| Tracing | `depth` field indents inner steps in logs |
| Failure | Parent fail-fast; no publish after failed research |
| Testing | Isolate sub-flows with minimal `WorkflowContext` |
| Depth limit | Prefer ≤ 2 levels of nesting |

Use nesting when phases are **owned by different teams**, **reused** across products, or **complex enough** to deserve their own orchestrator — not as an excuse for unlimited indirection.